In [ ]:
# Create RS Function BNBC2020-1 in ETABS
# Clones 'BNBC 2020' response spectrum function
# I=1.0 applied at RS load case level as SF = 1.0 x 9.81 = 9.81 m/s2
# Requirements: ETABS open with model | pip install comtypes

import comtypes.client

# -- 1. CONNECT -----------------------------------------------------------------
helper       = comtypes.client.CreateObject('ETABSv1.Helper')
helper       = helper.QueryInterface(comtypes.gen.ETABSv1.cHelper)
etabs_object = helper.GetObject('CSI.ETABS.API.ETABSObject')
model        = etabs_object.SapModel

ver      = model.GetVersion()
filename = model.GetModelFilename(True)
print(f'Connected : ETABS {ver[0]}')
print(f'Model     : {filename}')

# -- 2. SET UNITS ---------------------------------------------------------------
ret = model.SetPresentUnits(6)  # kN_m
assert ret == 0, f'SetPresentUnits failed: ret={ret}'
print('Units     : kN_m')

# -- 3. READ EXISTING RS FUNCTION TABLE -----------------------------------------
TABLE_NAME = 'Functions - Response Spectrum - ASCE7-05'

raw    = model.DatabaseTables.GetTableForDisplayArray(TABLE_NAME, [], 'All', 0, [], 0, [])
fields = [f for f in list(raw[2]) if f is not None]
n_rows = raw[3]
flat   = list(raw[4])
nf     = len(fields)
rows   = [{fields[j]: flat[i*nf+j] for j in range(nf)} for i in range(n_rows)]

print(f'\nRS function table: {n_rows} rows, fields: {fields}')

# Separate header row (has SsAndS1From populated) vs spectrum data rows
bnbc_header = [r for r in rows if r.get('Name') == 'BNBC 2020' and r.get('SsAndS1From') is not None]
bnbc_data   = [r for r in rows if r.get('Name') == 'BNBC 2020' and r.get('SsAndS1From') is None]

assert bnbc_header, "ERROR: source function 'BNBC 2020' not found in model"
bnbc_header = bnbc_header[0]

print(f'\nSource BNBC 2020 parameters:')
for k, v in bnbc_header.items():
    if v is not None:
        print(f'  {k:20s}: {v}')
print(f'  Spectrum data points: {len(bnbc_data)}')

# -- 4. BUILD NEW ROWS: clone BNBC 2020, rename to BNBC2020-1 ------------------
# Note: I factor is NOT stored in the RS function
# It is applied as scale factor in the RS load case: SF = I x g
#   Original  BNBC 2020   -> I=1.25 -> SF = 1.25 x 9.81 = 12.2625 m/s2
#   New  BNBC2020-1  -> I=1.00 -> SF = 1.00 x 9.81 =  9.8100 m/s2
NEW_FUNC = 'BNBC2020-1'

new_header         = dict(bnbc_header)
new_header['Name'] = NEW_FUNC
new_header['GUID'] = None       # ETABS assigns new GUID automatically

new_data_rows = []
for r in bnbc_data:
    nr = dict(r)
    nr['Name'] = NEW_FUNC
    new_data_rows.append(nr)

print(f'\nNew function: {NEW_FUNC}')
print(f'  Header row + {len(new_data_rows)} spectrum data points cloned from BNBC 2020')

# -- 5. REBUILD FLAT DATA (existing rows + new rows) ----------------------------
rows_out = list(rows) + [new_header] + new_data_rows
flat_out = tuple(r.get(f) for r in rows_out for f in fields)
print(f'  Total rows to write: {len(rows_out)}')

# -- 6. STAGE EDIT via SetTableForEditingArray ----------------------------------
# comtypes requires tuples, not lists, for SAFEARRAY input
ret_stage = model.DatabaseTables.SetTableForEditingArray(
    TABLE_NAME,
    0,
    tuple(fields),   # field names - must be tuple
    len(rows_out),   # row count
    flat_out         # flat data   - must be tuple
)
print(f'\nSetTableForEditingArray: {ret_stage[0]}  (expected 1 = staged OK)')
assert ret_stage[0] == 1, f'Staging failed: {ret_stage}'

# -- 7. APPLY EDITS via ApplyEditedTables ---------------------------------------
ret_apply = model.DatabaseTables.ApplyEditedTables(True)
n_fatal   = ret_apply[0]
n_warn    = ret_apply[1]
n_info    = ret_apply[2]
print(f'Fatal errors : {n_fatal}')
print(f'Warnings     : {n_warn}')
print(f'Info msgs    : {n_info}')
assert n_fatal == 0, 'ApplyEditedTables had fatal errors'

# -- 8. VERIFY ------------------------------------------------------------------
raw2    = model.DatabaseTables.GetTableForDisplayArray(TABLE_NAME, [], 'All', 0, [], 0, [])
fields2 = [f for f in list(raw2[2]) if f is not None]
n2      = raw2[3]
flat2   = list(raw2[4])
nf2     = len(fields2)
rows2   = [{fields2[j]: flat2[i*nf2+j] for j in range(nf2)} for i in range(n2)]

new_hdr  = [r for r in rows2 if r.get('Name') == NEW_FUNC and r.get('SsAndS1From') is not None]
new_data = [r for r in rows2 if r.get('Name') == NEW_FUNC and r.get('SsAndS1From') is None]

assert new_hdr, f"ERROR: '{NEW_FUNC}' not found after apply"
new_hdr = new_hdr[0]

print(f"\n'{NEW_FUNC}' confirmed in ETABS:")
for k, v in new_hdr.items():
    if v is not None:
        print(f'  {k:20s}: {v}')
print(f'  Spectrum data points: {len(new_data)}')

# Print T vs Sa table
print(f'\n  T (s)       Sa (g)')
print(f'  {"─"*22}')
all_pts = [new_hdr] + new_data
for r in all_pts:
    print(f'  {float(r["Period"]):8.4f}    {float(r["Value"]):8.6f}')

print(f'\nSUCCESS: {NEW_FUNC} created')
print(f'To use with I=1.0: assign to RS load case with SF = 1.0 x 9.81 = 9.81 m/s2')
print(f'Compare: BNBC 2020 with I=1.25 uses SF = 1.25 x 9.81 = 12.2625 m/s2')